# RNN and LSTM for NLP

Minimal PyTorch examples for sentiment analysis, machine translation, and text summarization. Models are not trained; token IDs are toy inputs.

## 1. Model definitions

In [ ]:
import torch
from torch import nn

VOCAB, EMBED, HIDDEN = 50, 16, 32

class SentimentModel(nn.Module):
    def __init__(self, cell='rnn'):
        super().__init__()
        R = nn.RNN if cell == 'rnn' else nn.LSTM
        self.cell = cell
        self.embed = nn.Embedding(VOCAB, EMBED)
        self.rnn = R(EMBED, HIDDEN, batch_first=True)
        self.fc = nn.Linear(HIDDEN, 2)

    def forward(self, x):
        _, state = self.rnn(self.embed(x))
        h = state[0] if self.cell == 'lstm' else state
        return self.fc(h[-1])

class Seq2Seq(nn.Module):
    def __init__(self, cell='rnn'):
        super().__init__()
        R = nn.RNN if cell == 'rnn' else nn.LSTM
        self.src_embed = nn.Embedding(VOCAB, EMBED)
        self.tgt_embed = nn.Embedding(VOCAB, EMBED)
        self.encoder = R(EMBED, HIDDEN, batch_first=True)
        self.decoder = R(EMBED, HIDDEN, batch_first=True)
        self.fc = nn.Linear(HIDDEN, VOCAB)

    def forward(self, src, tgt_in):
        _, state = self.encoder(self.src_embed(src))
        out, _ = self.decoder(self.tgt_embed(tgt_in), state)
        return self.fc(out)

## 2. Sentiment analysis

Returns two logits per sentence: negative and positive. Train on labelled reviews before prediction.

In [ ]:
reviews = torch.tensor([[2, 4, 5, 8], [3, 7, 2, 1]])
for cell in ('rnn', 'lstm'):
    model = SentimentModel(cell)
    print(cell, model(reviews).shape)

## 3. Machine translation

Encoder-decoder model maps source tokens to target-token logits. tgt_in is the shifted target sequence used for teacher forcing.

In [ ]:
source = torch.tensor([[2, 4, 5], [3, 8, 9]])
target_in = torch.tensor([[1, 7, 4], [1, 5, 6]])
for cell in ('rnn', 'lstm'):
    model = Seq2Seq(cell)
    print(cell, model(source, target_in).shape)

## 4. Text summarization

The same encoder-decoder structure predicts a shorter summary sequence. Train on article-summary pairs.

In [ ]:
article = torch.tensor([[4, 6, 8, 9, 3], [2, 5, 7, 6, 8]])
summary_in = torch.tensor([[1, 7], [1, 6]])
for cell in ('rnn', 'lstm'):
    model = Seq2Seq(cell)
    print(cell, model(article, summary_in).shape)

## Notes

- Install dependencies with pip install -r requirements.txt.
- Replace toy token IDs with a tokenizer and real datasets.
- Train using cross-entropy loss and shifted targets.
- This notebook checks model structure and tensor shapes only; it does not generate meaningful text.